# EAIM-Net v5 — Image Enhancement Tool

**Instructions:**
1. Run Cell 1 (Setup) once per session
2. Fill in your paths in Cell 2
3. Run Cell 3 to enhance and save
4. Run Cell 4 (optional) for a visual report

> Works with any JPG / PNG image. No zip extraction needed.


## Cell 1 — Setup
*Run once per session after connecting to Colab.*

In [ ]:
# ================================================================
# CELL 1 — SETUP  (run once per session)
# ================================================================
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
def pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
pip("lpips","scikit-image","tqdm","pyyaml","matplotlib")

import os, sys, warnings, random
import numpy as np
import torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as GS
from PIL import Image
from torchvision import transforms
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
warnings.filterwarnings("ignore")

# ── EDIT THESE TWO LINES ────────────────────────────────────────
DRIVE_ROOT  = "/content/drive/MyDrive/adaptive_enhancement_training"
COLAB_ROOT  = "/content/eaim_v5"
# ────────────────────────────────────────────────────────────────

DRIVE_CKPTS = f"{DRIVE_ROOT}/checkpoints_v5"
DRIVE_PRE   = f"{DRIVE_ROOT}/pretrained_filters"
DRIVE_RES   = f"{DRIVE_ROOT}/results_v5"
DRIVE_SRC   = f"{DRIVE_ROOT}/src_v5"

os.makedirs(COLAB_ROOT, exist_ok=True)
os.makedirs(DRIVE_RES,  exist_ok=True)
os.chdir(COLAB_ROOT)
sys.path.insert(0, COLAB_ROOT)

# Copy source files from Drive
import shutil
PY_FILES = [
    "afb_module.py","ess_module.py","epe_module.py","complete_model.py",
    "config.py","dataset.py","dataset_real_pairs.py","losses.py",
    "dataset_loader.py","dataset_loader_lazy.py",
]
print("Copying source files...")
for f in PY_FILES:
    src = os.path.join(DRIVE_SRC, f)
    dst = os.path.join(COLAB_ROOT, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  OK  {f}")
    elif os.path.exists(dst):
        print(f"  CACHE {f}")
    else:
        print(f"  MISS {f}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice : {DEVICE}")

# ── Load model (cached across cells) ────────────────────────────
from complete_model import AdaptiveEnhancementModel
from ess_module import FILTER_NAMES
from config import CONFIG

FILTER_NAMES_LIST = ["Low-light","Dehazing","Rain removal","Illum. norm","Glare red."]
WEATHER_NAMES = ["Clear","Rain","Fog","Light Snow","Glare"]
TIME_NAMES    = ["Dawn","Day","Dusk","Night"]
ILLUM_NAMES   = ["Low","Medium","High"]
FILTER_COLORS = ["#378ADD","#1D9E75","#EF9F27","#7F77DD","#D85A30"]

def load_model(ckpt_path):
    m = AdaptiveEnhancementModel(
        backbone            = CONFIG["model"]["backbone"],
        num_weather_classes = CONFIG["model"]["num_weather_classes"],
        num_time_classes    = CONFIG["model"]["num_time_classes"],
        num_illum_classes   = CONFIG["model"]["num_illum_classes"],
        feature_dim         = CONFIG["model"]["feature_dim"],
        num_filters         = CONFIG["model"]["num_filters"],
        pretrained          = False,
    ).to(DEVICE)
    ck = torch.load(ckpt_path, map_location=DEVICE)
    try:    m.load_state_dict(ck["model_state_dict"])
    except: m.load_state_dict(ck["model_state_dict"], strict=False)
    m.eval()
    return m

def enhance_image(model, inp_pil):
    IW, IH = inp_pil.size
    PW = ((IW+3)//4)*4; PH = ((IH+3)//4)*4
    pad = Image.new("RGB",(PW,PH)); pad.paste(inp_pil,(0,0))
    t = transforms.ToTensor()(pad).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(t)
    enh = out["enhanced"][0].cpu().numpy().transpose(1,2,0)[:IH,:IW]
    fw  = out["filter_weights"][0].cpu().numpy()
    H   = out["weight_entropy"][0].item()
    wl  = out["weather_logits"][0].argmax().item()
    tl  = out["time_logits"][0].argmax().item()
    il  = out["illum_logits"][0].argmax().item()
    return np.clip(enh,0,1), fw, H, wl, tl, il

print("\nSetup complete. Proceed to Cell 2.")


## Cell 2 — Set your paths
*Edit the paths below, then run Cell 3.*

In [ ]:
# ================================================================
# CELL 2 — CONFIGURE PATHS AND OPTIONS
# Edit these variables, then run Cell 3.
# ================================================================

# ── Input ────────────────────────────────────────────────────────
# Path to the image you want to enhance.
# Any JPG / PNG on Google Drive or local /content/ disk.
INPUT_IMAGE = "/content/drive/MyDrive/my_image.jpg"

# ── Output ───────────────────────────────────────────────────────
# Where to save the enhanced image.
# Folder is created automatically if it does not exist.
OUTPUT_IMAGE = "/content/drive/MyDrive/enhanced/my_image_enhanced.jpg"

# ── Checkpoint ───────────────────────────────────────────────────
# Use "best" for the best trained model.
# Options: "best" | "ep59" | "ep58" | "assembled"
CHECKPOINT = "best"

# ── Output quality (JPEG only, ignored for PNG) ──────────────────
# 95 = high quality, 85 = balanced, 75 = smaller file
JPEG_QUALITY = 92

# ── Extra outputs ─────────────────────────────────────────────────
SAVE_DIFF_MAP     = True   # save difference map (shows what changed)
SAVE_WEIGHTS_CHART = True  # save filter weights bar chart
SAVE_REPORT       = True   # save full comparison report (input|enhanced|GT)

# ── Batch mode (optional) ─────────────────────────────────────────
# To enhance a whole folder, set BATCH_INPUT_DIR and BATCH_OUTPUT_DIR.
# Leave as None to use single-image mode above.
BATCH_INPUT_DIR  = None   # e.g. "/content/drive/MyDrive/test_images/"
BATCH_OUTPUT_DIR = None   # e.g. "/content/drive/MyDrive/enhanced/"

# ── Resolve checkpoint path ───────────────────────────────────────
import glob as _gl
_CKPT_MAP = {
    "best":      os.path.join(DRIVE_CKPTS, "best_model.pth"),
    "ep59":      os.path.join(DRIVE_CKPTS, "checkpoint_epoch_0059.pth"),
    "ep58":      os.path.join(DRIVE_CKPTS, "checkpoint_epoch_0058.pth"),
    "assembled": os.path.join(DRIVE_CKPTS, "assembled_pretrained.pth"),
}
# Auto-find best if not explicit
if CHECKPOINT == "best":
    _b = _CKPT_MAP["best"]
    if not os.path.exists(_b):
        _eps = sorted(_gl.glob(os.path.join(DRIVE_CKPTS,"checkpoint_epoch_*.pth")))
        _CKPT_MAP["best"] = _eps[-1] if _eps else _CKPT_MAP["assembled"]

CKPT_PATH = _CKPT_MAP.get(CHECKPOINT, CHECKPOINT)

print(f"Input       : {INPUT_IMAGE}")
print(f"Output      : {OUTPUT_IMAGE}")
print(f"Checkpoint  : {os.path.basename(CKPT_PATH)}")
print(f"Batch mode  : {'ON  → ' + str(BATCH_INPUT_DIR) if BATCH_INPUT_DIR else 'OFF (single image)'}")
print()
if not os.path.exists(CKPT_PATH):
    print(f"WARNING: Checkpoint not found: {CKPT_PATH}")
else:
    sz = os.path.getsize(CKPT_PATH)/1e6
    print(f"Checkpoint found: {sz:.1f} MB")
print("\nReady. Run Cell 3 to enhance.")


## Cell 3 — Enhance and save
*Run after setting paths in Cell 2.*

In [ ]:
# ================================================================
# CELL 3 — ENHANCE IMAGE(S) AND SAVE
# ================================================================
import os, time
import numpy as np
from PIL import Image

assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}\nRun Cell 2 first."

# Load model
print("Loading model...")
MODEL = load_model(CKPT_PATH)
print(f"  OK — {sum(p.numel() for p in MODEL.parameters()):,} parameters")

def process_one(inp_path, out_path, label=""):
    """Enhance one image and save. Returns result dict."""
    assert os.path.exists(inp_path), f"Input not found: {inp_path}"
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)

    inp_pil = Image.open(inp_path).convert("RGB")
    inp_np  = np.array(inp_pil).astype(np.float32)/255.0

    t0 = time.time()
    enh_np, fw, H, wl, tl, il = enhance_image(MODEL, inp_pil)
    elapsed = time.time()-t0

    # Save enhanced image
    enh_pil = Image.fromarray((enh_np*255).astype(np.uint8))
    ext = os.path.splitext(out_path)[1].lower()
    if ext in [".jpg",".jpeg"]:
        enh_pil.save(out_path, quality=JPEG_QUALITY)
    else:
        enh_pil.save(out_path)

    # Save extras
    stem = os.path.splitext(out_path)[0]

    if SAVE_DIFF_MAP:
        diff = np.clip(np.abs(enh_np - inp_np)*5, 0, 1)
        Image.fromarray((diff*255).astype(np.uint8)).save(stem+"_diff.jpg", quality=90)

    if SAVE_WEIGHTS_CHART:
        fig, ax = plt.subplots(figsize=(8,3))
        bars = ax.bar(FILTER_NAMES_LIST, fw, color=FILTER_COLORS, width=0.6, edgecolor="white")
        ax.set_ylim(0,1); ax.set_ylabel("Blending weight")
        ax.set_title(f"ESS filter weights   H={H:.3f}   {WEATHER_NAMES[wl]}·{TIME_NAMES[tl]}·{ILLUM_NAMES[il]}", fontsize=10)
        for b,v in zip(bars,fw):
            ax.text(b.get_x()+b.get_width()/2, v+.02, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
        ax.grid(axis="y",alpha=0.3); ax.set_axisbelow(True)
        plt.tight_layout()
        fig.savefig(stem+"_weights.jpg", dpi=120, bbox_inches="tight")
        plt.close()

    return {"inp":inp_path,"out":out_path,"fw":fw,"H":H,
            "wl":wl,"tl":tl,"il":il,"elapsed":elapsed,"enh_np":enh_np,"inp_np":inp_np}

# ── Single image mode ─────────────────────────────────────────────
if BATCH_INPUT_DIR is None:
    print(f"\nEnhancing: {os.path.basename(INPUT_IMAGE)}")
    result = process_one(INPUT_IMAGE, OUTPUT_IMAGE)
    fw, H, wl, tl, il = result["fw"], result["H"], result["wl"], result["tl"], result["il"]

    print(f"\n{'='*52}")
    print(f"  Enhancement complete")
    print(f"{'='*52}")
    print(f"  Saved         : {OUTPUT_IMAGE}")
    print(f"  Time          : {result['elapsed']:.1f}s")
    print(f"  EPE           : {WEATHER_NAMES[wl]} · {TIME_NAMES[tl]} · Illum={ILLUM_NAMES[il]}")
    print(f"  Entropy H     : {H:.4f}  ({'genuine blend' if H>0.20 else 'low — check model'})")
    print()
    print("  Filter weights:")
    for name, w in zip(FILTER_NAMES_LIST, fw):
        bar = "█" * int(w*30)
        print(f"  {name:<22} {bar:<30} {w:.4f}")
    print(f"{'='*52}")
    if SAVE_DIFF_MAP:
        print(f"  Diff map      : {os.path.splitext(OUTPUT_IMAGE)[0]}_diff.jpg")
    if SAVE_WEIGHTS_CHART:
        print(f"  Weights chart : {os.path.splitext(OUTPUT_IMAGE)[0]}_weights.jpg")
    print("\nRun Cell 4 to see visual comparison.")

# ── Batch mode ────────────────────────────────────────────────────
else:
    assert os.path.isdir(BATCH_INPUT_DIR), f"Not a folder: {BATCH_INPUT_DIR}"
    os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)
    exts = (".jpg",".jpeg",".png",".bmp")
    files = sorted([f for f in os.listdir(BATCH_INPUT_DIR) if f.lower().endswith(exts)])
    print(f"\nBatch mode: {len(files)} images in {BATCH_INPUT_DIR}")
    results = []
    for idx, fname in enumerate(files):
        inp = os.path.join(BATCH_INPUT_DIR,  fname)
        out = os.path.join(BATCH_OUTPUT_DIR, fname)
        print(f"  [{idx+1:3d}/{len(files)}] {fname}...", end="", flush=True)
        try:
            r = process_one(inp, out)
            print(f" done ({r['elapsed']:.1f}s)  H={r['H']:.3f}  {WEATHER_NAMES[r['wl']]}")
            results.append(r)
        except Exception as e:
            print(f" FAILED: {e}")
    print(f"\nBatch complete: {len(results)}/{len(files)} enhanced")
    print(f"Output folder: {BATCH_OUTPUT_DIR}")


## Cell 4 — Visual report
*Shows comparison figures inline. Run after Cell 3.*

In [ ]:
# ================================================================
# CELL 4 — VISUAL COMPARISON REPORT
# ================================================================
# Shows: Input | Enhanced | Difference map | Filter weights
# Also saves a combined report image to the output folder.
# ================================================================

if BATCH_INPUT_DIR is not None and results:
    # For batch: pick a random result to visualise
    result = random.choice(results)
    print(f"Showing random batch result: {os.path.basename(result['inp'])}")

assert 'result' in dir() or 'results' in dir(), "Run Cell 3 first."
if 'result' not in dir(): result = results[0]

enh_np = result["enh_np"]
inp_np = result["inp_np"]
fw     = result["fw"]
H      = result["H"]
wl, tl, il = result["wl"], result["tl"], result["il"]

# ── Figure 1: Comparison ──────────────────────────────────────────
fig1, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor="white")
axes[0].imshow(np.clip(inp_np,0,1));  axes[0].axis("off")
axes[0].set_title(f"Input\n{os.path.basename(result['inp'])}", fontsize=9)

axes[1].imshow(np.clip(enh_np,0,1)); axes[1].axis("off")
axes[1].set_title(
    f"Enhanced\nEPE: {WEATHER_NAMES[wl]} · {TIME_NAMES[tl]} · {ILLUM_NAMES[il]}   H={H:.3f}",
    fontsize=9, color="#1a7a1a")

diff = np.clip(np.abs(enh_np - inp_np)*5, 0, 1)
im   = axes[2].imshow(diff, cmap="hot"); axes[2].axis("off")
axes[2].set_title("Difference map (×5)\nBright = most changed", fontsize=9)
plt.colorbar(im, ax=axes[2], shrink=0.8)

fig1.suptitle("EAIM-Net v5 — Enhancement Result", fontsize=12, fontweight="bold")
plt.tight_layout()
rep_path = os.path.splitext(result["out"])[0] + "_report.jpg"
if SAVE_REPORT:
    fig1.savefig(rep_path, dpi=130, bbox_inches="tight")
    print(f"Report saved: {rep_path}")
plt.show()

# ── Figure 2: Filter weights ──────────────────────────────────────
fig2, ax = plt.subplots(figsize=(9, 3.5), facecolor="white")
bars = ax.bar(FILTER_NAMES_LIST, fw, color=FILTER_COLORS, width=0.55, edgecolor="white")
ax.set_ylim(0, 1); ax.set_ylabel("Blending weight")
ax.set_title(f"ESS filter weights   H={H:.4f}   τ={MODEL.ess.tau.item():.4f}", fontsize=10)
ax.axhline(0.20, color="gray", lw=0.8, ls="--", alpha=0.6, label="H>0.20 threshold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
for b, v in zip(bars, fw):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.3f}",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

# ── Summary ───────────────────────────────────────────────────────
print()
print("="*52)
print("  Visual report complete")
print("="*52)
print(f"  Input    : {os.path.basename(result['inp'])}")
print(f"  Output   : {result['out']}")
print(f"  EPE      : {WEATHER_NAMES[wl]} · {TIME_NAMES[tl]} · {ILLUM_NAMES[il]}")
print(f"  H        : {H:.4f}")
dominant = FILTER_NAMES_LIST[fw.argmax()]
print(f"  Dominant : {dominant} ({fw.max():.3f})")
print("="*52)


## Cell 4b — Interactive side-by-side slider
*Drag the divider to compare input vs enhanced. Run after Cell 3.*

In [ ]:
# ================================================================
# CELL 4b — INTERACTIVE SIDE-BY-SIDE SLIDER PANEL
# ================================================================
# Shows input and enhanced image with a draggable split slider.
# Move the slider left/right to compare the two images.
# Also shows EPE prediction and filter weights live.
# ================================================================

import base64, io, json as _json
from IPython.display import display, HTML
import numpy as np
from PIL import Image

assert 'result' in dir() or 'results' in dir(), "Run Cell 3 first."
if 'result' not in dir(): result = results[0]

enh_np = result["enh_np"]
inp_np = result["inp_np"]
fw     = result["fw"]
H      = result["H"]
wl, tl, il = result["wl"], result["tl"], result["il"]

def np_to_b64(arr):
    arr_uint8 = (np.clip(arr,0,1)*255).astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(arr_uint8).save(buf, format="JPEG", quality=92)
    return base64.b64encode(buf.getvalue()).decode()

inp_b64 = np_to_b64(inp_np)
enh_b64 = np_to_b64(enh_np)

fw_json       = _json.dumps(fw.tolist())
names_json    = _json.dumps(FILTER_NAMES_LIST)
colors_json   = _json.dumps(FILTER_COLORS)
weather_label = WEATHER_NAMES[wl]
time_label    = TIME_NAMES[tl]
illum_label   = ILLUM_NAMES[il]
fname         = __import__('os').path.basename(result['inp'])

html = f"""
<style>
  .panel-wrap {{
    font-family: system-ui, sans-serif;
    font-size: 13px;
    color: #222;
    background: #fafafa;
    border: 1px solid #e0e0e0;
    border-radius: 12px;
    overflow: hidden;
    max-width: 860px;
    margin: 0 auto;
  }}
  .panel-header {{
    background: #1B3A6B;
    color: #fff;
    padding: 12px 18px;
    display: flex;
    align-items: center;
    justify-content: space-between;
  }}
  .panel-header h3 {{ margin:0; font-size:15px; font-weight:500; }}
  .panel-header .badge {{
    background: rgba(255,255,255,0.2);
    border-radius: 6px;
    padding: 3px 10px;
    font-size: 12px;
  }}
  .slider-section {{
    position: relative;
    width: 100%;
    user-select: none;
    overflow: hidden;
    cursor: col-resize;
    background: #000;
  }}
  .slider-section img {{
    display: block;
    width: 100%;
    height: auto;
    max-height: 420px;
    object-fit: contain;
  }}
  .img-before {{
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 100%;
    overflow: hidden;
  }}
  .img-before img {{
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: auto;
    max-height: 420px;
    object-fit: contain;
  }}
  .divider-line {{
    position: absolute;
    top: 0; bottom: 0;
    width: 2px;
    background: #fff;
    box-shadow: 0 0 6px rgba(0,0,0,0.6);
    cursor: col-resize;
    z-index: 10;
  }}
  .divider-handle {{
    position: absolute;
    top: 50%;
    left: 50%;
    transform: translate(-50%,-50%);
    width: 36px; height: 36px;
    background: #fff;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    box-shadow: 0 2px 8px rgba(0,0,0,0.35);
    font-size: 14px;
    color: #1B3A6B;
    font-weight: 600;
    cursor: col-resize;
    white-space: nowrap;
  }}
  .img-label {{
    position: absolute;
    top: 10px;
    padding: 3px 10px;
    border-radius: 6px;
    font-size: 12px;
    font-weight: 500;
    color: #fff;
    pointer-events: none;
  }}
  .label-before {{ left:10px; background:rgba(0,0,0,0.55); }}
  .label-after  {{ right:10px; background:rgba(27,58,107,0.85); }}
  .info-grid {{
    display: grid;
    grid-template-columns: repeat(4,1fr);
    gap: 8px;
    padding: 12px 16px;
    border-top: 1px solid #eee;
    background: #fff;
  }}
  .info-card {{
    background: #f5f7fa;
    border-radius: 8px;
    padding: 8px 10px;
  }}
  .info-card .lbl {{ font-size:11px; color:#888; margin:0 0 3px; }}
  .info-card .val {{ font-size:15px; font-weight:500; color:#1B3A6B; margin:0; }}
  .weights-section {{
    padding: 12px 16px 16px;
    background: #fff;
    border-top: 1px solid #eee;
  }}
  .weights-title {{ font-size:12px; color:#888; margin:0 0 8px; }}
  .wbar-row {{
    display: grid;
    grid-template-columns: 110px 1fr 42px;
    align-items: center;
    gap: 8px;
    margin-bottom: 5px;
  }}
  .wbar-name {{ font-size:12px; color:#444; }}
  .wbar-track {{
    height: 7px;
    background: #eee;
    border-radius: 4px;
    overflow: hidden;
  }}
  .wbar-fill {{
    height: 100%;
    border-radius: 4px;
    transition: width .4s;
  }}
  .wbar-pct {{ font-size:11px; color:#888; text-align:right; }}
  .fname-row {{
    padding: 6px 16px;
    font-size:11px;
    color:#aaa;
    background:#fff;
    border-top: 1px solid #eee;
  }}
  .drag-hint {{
    position: absolute;
    bottom: 8px;
    left: 50%;
    transform: translateX(-50%);
    background: rgba(0,0,0,0.5);
    color: #fff;
    font-size: 11px;
    padding: 3px 10px;
    border-radius: 20px;
    pointer-events: none;
    opacity: 1;
    transition: opacity 1s;
  }}
</style>

<div class="panel-wrap">
  <div class="panel-header">
    <h3>EAIM-Net v5 — Enhancement Comparison</h3>
    <span class="badge">{weather_label} &middot; {time_label} &middot; {illum_label}</span>
  </div>

  <div class="slider-section" id="sliderWrap">
    <img id="imgAfter" src="data:image/jpeg;base64,{enh_b64}" alt="Enhanced image" />
    <div class="img-before" id="beforeDiv">
      <img id="imgBefore" src="data:image/jpeg;base64,{inp_b64}" alt="Input image" />
    </div>
    <div class="divider-line" id="divLine">
      <div class="divider-handle">&#8596;</div>
    </div>
    <span class="img-label label-before">Input</span>
    <span class="img-label label-after">Enhanced</span>
    <div class="drag-hint" id="dragHint">&#8592; drag to compare &#8594;</div>
  </div>

  <div class="info-grid">
    <div class="info-card">
      <p class="lbl">Weather</p>
      <p class="val">{weather_label}</p>
    </div>
    <div class="info-card">
      <p class="lbl">Time of day</p>
      <p class="val">{time_label}</p>
    </div>
    <div class="info-card">
      <p class="lbl">Illumination</p>
      <p class="val">{illum_label}</p>
    </div>
    <div class="info-card">
      <p class="lbl">Entropy H</p>
      <p class="val">{H:.4f}</p>
    </div>
  </div>

  <div class="weights-section">
    <p class="weights-title">Filter blending weights</p>
    <div id="wbars"></div>
  </div>

  <div class="fname-row">{fname}</div>
</div>

<script>
(function() {{
  const fw      = {fw_json};
  const names   = {names_json};
  const colors  = {colors_json};

  // Build weight bars
  const container = document.getElementById('wbars');
  names.forEach((name, k) => {{
    const pct = (fw[k]*100).toFixed(1);
    container.innerHTML += `
      <div class="wbar-row">
        <span class="wbar-name">${{name}}</span>
        <div class="wbar-track">
          <div class="wbar-fill" style="width:${{pct}}%;background:${{colors[k]}}"></div>
        </div>
        <span class="wbar-pct">${{pct}}%</span>
      </div>`;
  }});

  // Slider logic
  const wrap    = document.getElementById('sliderWrap');
  const before  = document.getElementById('beforeDiv');
  const divLine = document.getElementById('divLine');
  const hint    = document.getElementById('dragHint');
  let dragging  = false;
  let pos       = 50;  // percent

  function setPos(pct) {{
    pct = Math.max(2, Math.min(98, pct));
    pos = pct;
    before.style.width   = pct + '%';
    divLine.style.left   = pct + '%';
  }}

  setPos(50);

  function getX(e) {{
    const rect = wrap.getBoundingClientRect();
    const clientX = e.touches ? e.touches[0].clientX : e.clientX;
    return ((clientX - rect.left) / rect.width) * 100;
  }}

  wrap.addEventListener('mousedown',  e => {{ dragging=true; setPos(getX(e)); hint.style.opacity='0'; }});
  wrap.addEventListener('touchstart', e => {{ dragging=true; setPos(getX(e)); hint.style.opacity='0'; }}, {{passive:true}});
  document.addEventListener('mousemove',  e => {{ if(dragging) setPos(getX(e)); }});
  document.addEventListener('touchmove',  e => {{ if(dragging) setPos(getX(e)); }}, {{passive:true}});
  document.addEventListener('mouseup',   () => dragging=false);
  document.addEventListener('touchend',  () => dragging=false);

  // Hide hint after 3s
  setTimeout(() => hint.style.opacity='0', 3000);
}})();
</script>
"""

display(HTML(html))
print(f"\nDrag the slider left/right to compare input vs enhanced.")
print(f"EPE: {weather_label} · {time_label} · Illum={illum_label}   H={H:.4f}")


## Cell 5 — Batch visual grid
*Only for batch mode. Shows a grid of all enhanced images.*

In [ ]:
# ================================================================
# CELL 5 — BATCH VISUAL GRID (batch mode only)
# Shows up to 12 input→enhanced pairs in a grid.
# ================================================================
if BATCH_INPUT_DIR is None:
    print("Batch mode is OFF. Set BATCH_INPUT_DIR in Cell 2.")
else:
    n_show = min(len(results), 12)
    sample = random.sample(results, n_show)
    ncols  = 4
    nrows  = n_show * 2 // ncols + (1 if (n_show*2) % ncols else 0)

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3.5), facecolor="white")
    axes = axes.flatten()

    for idx, r in enumerate(sample):
        r2 = idx * 2
        axes[r2].imshow(np.clip(r["inp_np"],0,1)); axes[r2].axis("off")
        axes[r2].set_title(f"Input\n{os.path.basename(r['inp'])[:18]}", fontsize=7)
        axes[r2+1].imshow(np.clip(r["enh_np"],0,1)); axes[r2+1].axis("off")
        axes[r2+1].set_title(
            f"Enhanced\n{WEATHER_NAMES[r['wl']]}  H={r['H']:.2f}", fontsize=7, color="#1a7a1a")

    for ax in axes[n_show*2:]: ax.axis("off")
    fig.suptitle(f"EAIM-Net v5 — Batch Results (showing {n_show}/{len(results)})",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    grid_path = os.path.join(BATCH_OUTPUT_DIR, "_batch_grid.jpg")
    fig.savefig(grid_path, dpi=110, bbox_inches="tight")
    plt.show()
    print(f"Grid saved: {grid_path}")
